# 2 Clients, 2 Different Datasets (CIFAR-10 + STL-10) — Comparing 4 Federated Algorithms

**Setup:** exactly 2 clients.
- **Client 1** trains on **CIFAR-10**.
- **Client 2** trains on **STL-10** (a completely different dataset — different source, different
  resolution, different image statistics).

Both datasets are restricted to the **9 classes they have in common** (airplane, automobile/car, bird,
cat, deer, dog, horse, ship, truck) and remapped to one shared label space, so the two clients are
predicting over the exact same classes — this is what makes federating them meaningful at all.

Both clients use the **same model architecture**, so weight-averaging algorithms remain valid.

**Algorithms compared:**
| # | Algorithm | Why it's here |
|---|---|---|
| 1 | **FedAvg** | Baseline — plain weighted average of the 2 clients' weights |
| 2 | **FedProx** | Adds a penalty keeping each client close to the global model — built specifically for this kind of heterogeneous-data situation |
| 3 | **FedAdam** | Server-side adaptive optimizer (Adam) applied to the aggregated update — usually the strongest of the FedOpt family on skewed/heterogeneous data, so this is my pick for "likely highest accuracy" |
| 4 | **FedDistillation (FedDF)** | Clients share predictions instead of weights, combined via a small shared public set — doesn't rely on weight averaging at all |

With only 2 clients this is a small-scale test, but it isolates exactly the effect of **dataset-level
heterogeneity** (not model or feature heterogeneity) on each algorithm.

---
## Section 1: Imports & Device Setup

In [1]:
import sys
!{sys.executable} -m pip install -q torch torchvision

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
import torchvision, torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(42); torch.manual_seed(42)
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

Device  : cpu
PyTorch : 2.11.0+cpu


---
## Section 2: Configuration

In [3]:
CONFIG = {
    'num_rounds'     : 10,
    'local_epochs'   : 5,
    'batch_size'     : 64,
    'learning_rate'  : 0.001,
    'temperature'    : 4.0,   # distillation temperature
    'num_public'     : 300,   # shared public proxy samples for FedDistillation
    'distill_epochs' : 10,
    'mu_prox'        : 0.01,  # FedProx proximal strength
    'server_lr'      : 0.1,   # server-side learning rate for FedAdam
    'beta1'          : 0.9,
    'beta2'          : 0.99,
    'tau'            : 1e-3,
}
# Shared 9-class unified label space (classes present in BOTH CIFAR-10 and STL-10)
UNIFIED_CLASSES = ['Airplane','Automobile','Bird','Cat','Deer','Dog','Horse','Ship','Truck']
# CIFAR-10 target (0-9) -> unified id, or None if no STL-10 counterpart (frog)
CIFAR_TO_UNIFIED = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:None, 7:6, 8:7, 9:8}
# torchvision STL10 classes: ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
STL_TO_UNIFIED   = {0:0, 1:2, 2:1, 3:3, 4:4, 5:5, 6:6, 7:None, 8:7, 9:8}

print('Configuration loaded')
for k, v in CONFIG.items(): print(f'   {k}: {v}')
print('Unified shared label space (9 classes):', UNIFIED_CLASSES)
print('CIFAR-10 "frog" and STL-10 "monkey" are dropped (no shared counterpart).')

Configuration loaded
   num_rounds: 10
   local_epochs: 5
   batch_size: 64
   learning_rate: 0.001
   temperature: 4.0
   num_public: 300
   distill_epochs: 10
   mu_prox: 0.01
   server_lr: 0.1
   beta1: 0.9
   beta2: 0.99
   tau: 0.001
Unified shared label space (9 classes): ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 'Dog', 'Horse', 'Ship', 'Truck']
CIFAR-10 "frog" and STL-10 "monkey" are dropped (no shared counterpart).


---
## Section 3: Load CIFAR-10 (Client 1) and STL-10 (Client 2), remapped to shared labels

In [ ]:
transform_cifar_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_cifar_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_stl_train = transforms.Compose([
    transforms.Resize((32, 32)),          # match CIFAR-10 resolution so the same CNN works
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])
transform_stl_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])

class RemappedDataset(torch.utils.data.Dataset):
    """Filters a dataset down to classes present in `label_map` and remaps labels to the unified space."""
    def __init__(self, base_dataset, label_map, transform, targets):
        self.base, self.label_map, self.transform = base_dataset, label_map, transform
        self.indices = [i for i, t in enumerate(targets) if label_map.get(int(t)) is not None]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        return self.transform(img), self.label_map[int(label)]

# --- Client 1: CIFAR-10 ---
cifar_train_raw = torchvision.datasets.CIFAR10('.', train=True,  download=True)
cifar_test_raw  = torchvision.datasets.CIFAR10('.', train=False, download=True)
cifar_train_ds  = RemappedDataset(cifar_train_raw, CIFAR_TO_UNIFIED, transform_cifar_train, cifar_train_raw.targets)
cifar_test_ds   = RemappedDataset(cifar_test_raw,  CIFAR_TO_UNIFIED, transform_cifar_test,  cifar_test_raw.targets)

# --- Client 2: STL-10 (NOTE: first download is ~2.5GB) ---
stl_train_raw = torchvision.datasets.STL10('.', split='train', download=True)
stl_test_raw  = torchvision.datasets.STL10('.', split='test',  download=True)
stl_train_ds  = RemappedDataset(stl_train_raw, STL_TO_UNIFIED, transform_stl_train, stl_train_raw.labels)
stl_test_ds   = RemappedDataset(stl_test_raw,  STL_TO_UNIFIED, transform_stl_test,  stl_test_raw.labels)

client_datasets = [cifar_train_ds, stl_train_ds]
client_names    = ['Client 1 (CIFAR-10)', 'Client 2 (STL-10)']
client_sizes    = [len(cifar_train_ds), len(stl_train_ds)]

client_loaders = [DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True) for ds in client_datasets]

# Public proxy set for FedDistillation: small shared sample pooled from BOTH clients' test data
public_pool   = ConcatDataset([Subset(cifar_test_ds, range(CONFIG['num_public']//2)),
                                Subset(stl_test_ds,   range(CONFIG['num_public']//2))])
public_loader = DataLoader(public_pool, batch_size=len(public_pool), shuffle=False)

# Evaluation: combined test set (both datasets) + each dataset separately
test_loader        = DataLoader(ConcatDataset([cifar_test_ds, stl_test_ds]), batch_size=256, shuffle=False)
test_loader_cifar  = DataLoader(cifar_test_ds, batch_size=256, shuffle=False)
test_loader_stl    = DataLoader(stl_test_ds,   batch_size=256, shuffle=False)

for name, size in zip(client_names, client_sizes):
    print(f'   {name}: {size} training samples')
print(f'Combined test set: {len(cifar_test_ds)} CIFAR-10 + {len(stl_test_ds)} STL-10 = {len(cifar_test_ds)+len(stl_test_ds)} images')
print(f'Public proxy set for FedDistillation: {len(public_pool)} samples')

100%|██████████| 170M/170M [34:54<00:00, 81.4kB/s]
100%|██████████| 2.64G/2.64G [06:07<00:00, 7.19MB/s]


   Client 1 (CIFAR-10): 45000 training samples
   Client 2 (STL-10): 4500 training samples
Combined test set: 9000 CIFAR-10 + 7200 STL-10 = 16200 images
Public proxy set for FedDistillation: 300 samples


---
## Section 4: Model Architecture (same for both clients)

In [ ]:
class UnifiedCNN(nn.Module):
    """Same architecture for both clients - 9-way head for the shared label space."""
    def __init__(self, num_classes=len(UNIFIED_CLASSES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

n_params = sum(p.numel() for p in UnifiedCNN().parameters())
print(f'UnifiedCNN: {n_params:,} parameters (identical for both clients, all 4 algorithms)')

UnifiedCNN: 544,969 parameters (identical for both clients, all 4 algorithms)


---
## Section 5: Helper Functions

In [ ]:
def train_local(model, loader, optimizer, epochs, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            ce(model(imgs), labels).backward()
            optimizer.step()

def train_local_fedprox(model, loader, optimizer, epochs, device, global_state, mu):
    model.train()
    ce = nn.CrossEntropyLoss()
    global_params = {k: v.clone().detach().to(device) for k, v in global_state.items()}
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            prox = sum(((p - global_params[n]) ** 2).sum()
                       for n, p in model.named_parameters() if n in global_params)
            (loss + (mu / 2) * prox).backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def get_soft_preds(model, public_X, device):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(public_X.to(device)), dim=1).cpu()

def weighted_avg_state(models, sizes):
    total = sum(sizes)
    keys = models[0].state_dict().keys()
    return {k: sum(models[i].state_dict()[k].float() * (sizes[i] / total) for i in range(len(models)))
            for k in keys}

def fedopt_server_update(global_model, avg_state, server_state, algo, cfg):
    """FedOpt-style server update (Reddi et al., 2021). algo in {'fedavg','fedadam'}."""
    g_state = global_model.state_dict()
    new_state = {}
    for key in g_state:
        delta = avg_state[key].float() - g_state[key].float()
        if algo == 'fedavg':
            new_state[key] = g_state[key].float() + delta
            continue
        if key not in server_state:
            server_state[key] = {'m': torch.zeros_like(delta), 'v': torch.full_like(delta, cfg['tau'] ** 2)}
        m, v = server_state[key]['m'], server_state[key]['v']
        m = cfg['beta1'] * m + (1 - cfg['beta1']) * delta
        v = cfg['beta2'] * v + (1 - cfg['beta2']) * (delta ** 2)
        server_state[key] = {'m': m, 'v': v}
        new_state[key] = g_state[key].float() + cfg['server_lr'] * m / (torch.sqrt(v) + cfg['tau'])
    return new_state

print('Helper functions ready')

Helper functions ready


---
## Section 6: Run FedAvg, FedProx, FedAdam

In [ ]:
def run_weight_based_algo(algo_name, use_prox=False):
    """Runs FedAvg / FedProx / FedAdam on the 2 clients (CIFAR-10 + STL-10, shared labels)."""
    label = 'FedProx' if use_prox else algo_name.upper()
    print('='*60); print(f'  {label}'); print('='*60)

    clients = [UnifiedCNN().to(DEVICE) for _ in range(2)]
    opts    = [optim.Adam(m.parameters(), lr=CONFIG['learning_rate']) for m in clients]
    global_model = UnifiedCNN().to(DEVICE)
    server_state = {}
    history = {'round': [], 'global_acc': [], 'acc_cifar': [], 'acc_stl': []}

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        g_state = global_model.state_dict()
        for c in clients:
            c.load_state_dict(g_state)

        for c, opt, loader in zip(clients, opts, client_loaders):
            if use_prox:
                train_local_fedprox(c, loader, opt, CONFIG['local_epochs'], DEVICE, g_state, CONFIG['mu_prox'])
            else:
                train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

        avg_state = weighted_avg_state(clients, client_sizes)
        new_state = fedopt_server_update(global_model, avg_state, server_state, algo_name, CONFIG)
        global_model.load_state_dict(new_state)

        g_acc  = evaluate_model(global_model, test_loader, DEVICE)
        c_acc  = evaluate_model(global_model, test_loader_cifar, DEVICE)
        s_acc  = evaluate_model(global_model, test_loader_stl, DEVICE)
        history['round'].append(rnd); history['global_acc'].append(g_acc)
        history['acc_cifar'].append(c_acc); history['acc_stl'].append(s_acc)
        print(f'  Round {rnd:02d} | Combined: {g_acc:.4f} | CIFAR-10: {c_acc:.4f} | STL-10: {s_acc:.4f}')
    print(f'Final combined accuracy: {history["global_acc"][-1]:.4f}\n')
    return history

# 1. FedAvg
history_fedavg  = run_weight_based_algo('fedavg')
# 2. FedProx (aggregation = plain FedAvg update, local objective has the proximal penalty)
history_fedprox = run_weight_based_algo('fedavg', use_prox=True)
# 3. FedAdam (server-side adaptive optimizer -> expected to handle this 2-client domain-shift best)
history_fedadam = run_weight_based_algo('fedadam')

  FEDAVG
  Round 01 | Combined: 0.6724 | CIFAR-10: 0.7211 | STL-10: 0.6115
  Round 02 | Combined: 0.6928 | CIFAR-10: 0.7418 | STL-10: 0.6315
  Round 03 | Combined: 0.7167 | CIFAR-10: 0.7640 | STL-10: 0.6576
  Round 04 | Combined: 0.7204 | CIFAR-10: 0.7721 | STL-10: 0.6558
  Round 05 | Combined: 0.7260 | CIFAR-10: 0.7833 | STL-10: 0.6544
  Round 06 | Combined: 0.7364 | CIFAR-10: 0.7918 | STL-10: 0.6672
  Round 07 | Combined: 0.7408 | CIFAR-10: 0.7961 | STL-10: 0.6717
  Round 08 | Combined: 0.7418 | CIFAR-10: 0.7951 | STL-10: 0.6751
  Round 09 | Combined: 0.7516 | CIFAR-10: 0.8074 | STL-10: 0.6818
  Round 10 | Combined: 0.7459 | CIFAR-10: 0.8064 | STL-10: 0.6703
Final combined accuracy: 0.7459

  FedProx
  Round 01 | Combined: 0.5744 | CIFAR-10: 0.6106 | STL-10: 0.5292
  Round 02 | Combined: 0.6524 | CIFAR-10: 0.6932 | STL-10: 0.6014
  Round 03 | Combined: 0.6682 | CIFAR-10: 0.7137 | STL-10: 0.6114
  Round 04 | Combined: 0.6551 | CIFAR-10: 0.6927 | STL-10: 0.6082
  Round 05 | Combined: 0

---
## Section 7: Run FedDistillation (FedDF)

In [ ]:
# 4. FedDistillation (FedDF) - clients share PREDICTIONS on a small shared public proxy set, not weights.
print('='*60); print('  FEDDISTILLATION (FedDF)'); print('='*60)

clients_fd = [UnifiedCNN().to(DEVICE) for _ in range(2)]
opts_fd    = [optim.Adam(m.parameters(), lr=CONFIG['learning_rate']) for m in clients_fd]
global_fd  = UnifiedCNN().to(DEVICE)
public_X_fd, _ = next(iter(public_loader))
history_feddistill = {'round': [], 'global_acc': [], 'acc_cifar': [], 'acc_stl': []}

for rnd in range(1, CONFIG['num_rounds'] + 1):
    # 1) each client trains locally on its own private dataset only
    for c, opt, loader in zip(clients_fd, opts_fd, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

    # 2) server collects each client's SOFT PREDICTIONS on the shared public proxy set
    client_soft_preds = [get_soft_preds(c, public_X_fd, DEVICE) for c in clients_fd]
    weights = np.array(client_sizes) / sum(client_sizes)
    ensemble_soft = sum(w * p for w, p in zip(weights, client_soft_preds))   # weighted ensemble = teacher

    # 3) global model distills the ensemble's knowledge (no weight averaging at all)
    global_opt = optim.Adam(global_fd.parameters(), lr=CONFIG['learning_rate'])
    for _ in range(CONFIG['distill_epochs']):
        global_opt.zero_grad()
        logits = global_fd(public_X_fd.to(DEVICE))
        T = CONFIG['temperature']
        s_log  = F.log_softmax(logits / T, dim=1)
        t_soft = F.softmax(ensemble_soft.to(DEVICE) / T, dim=1)
        (nn.KLDivLoss(reduction='batchmean')(s_log, t_soft) * (T ** 2)).backward()
        global_opt.step()

    # 4) broadcast the newly-distilled global model back down so next round's local training starts from it
    for c in clients_fd:
        c.load_state_dict(global_fd.state_dict())

    g_acc = evaluate_model(global_fd, test_loader, DEVICE)
    c_acc = evaluate_model(global_fd, test_loader_cifar, DEVICE)
    s_acc = evaluate_model(global_fd, test_loader_stl, DEVICE)
    history_feddistill['round'].append(rnd); history_feddistill['global_acc'].append(g_acc)
    history_feddistill['acc_cifar'].append(c_acc); history_feddistill['acc_stl'].append(s_acc)
    print(f'  Round {rnd:02d} | Combined: {g_acc:.4f} | CIFAR-10: {c_acc:.4f} | STL-10: {s_acc:.4f}')

print(f'Final combined accuracy: {history_feddistill["global_acc"][-1]:.4f}')

---
## Section 8: Compare All 4 Algorithms

In [ ]:
results = {
    'FedAvg'          : history_fedavg,
    'FedProx'         : history_fedprox,
    'FedAdam'         : history_fedadam,
    'FedDistillation' : history_feddistill,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('2 Clients, 2 Datasets (CIFAR-10 + STL-10, shared labels): 4 Algorithms Compared',
             fontsize=12, fontweight='bold')

palette = plt.cm.tab10(np.linspace(0, 1, 4))
for (name, hist), color in zip(results.items(), palette):
    axes[0].plot(hist['round'], hist['global_acc'], '-o', lw=2, ms=5, color=color, label=name)
axes[0].set(xlabel='Round', ylabel='Combined Test Accuracy', title='Accuracy over Rounds')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

final_accs = [hist['global_acc'][-1] for hist in results.values()]
bars = axes[1].bar(list(results.keys()), final_accs, color=palette)
axes[1].set(ylabel='Final Combined Test Accuracy', title='Final Accuracy Comparison')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(alpha=0.3, axis='y')
for b, a in zip(bars, final_accs):
    axes[1].text(b.get_x() + b.get_width()/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('two_client_dataset_algo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved as two_client_dataset_algo_comparison.png')

print('\nFinal ranking (combined test accuracy):')
for name, acc in sorted(zip(results.keys(), final_accs), key=lambda x: -x[1]):
    print(f'   {name:18s}: {acc:.4f}')

print('\nPer-dataset breakdown (final round):')
for name, hist in results.items():
    print(f'   {name:18s}: CIFAR-10 = {hist["acc_cifar"][-1]:.4f} | STL-10 = {hist["acc_stl"][-1]:.4f}')

---
## Section 9: Summary

**Setup recap:** exactly 2 clients — Client 1 on CIFAR-10, Client 2 on STL-10 — both restricted to the
9 classes they share, remapped to one label space, both using the identical `UnifiedCNN` architecture.

**Why FedAdam was picked as the "likely best" 4th algorithm:** with only 2 clients whose data comes from
genuinely different distributions (different camera sources, different resolutions before resizing), the
raw pseudo-gradient direction that FedAvg produces can be noisy. FedAdam's server-side momentum + adaptive
scaling smooths that out, which is generally the strongest performer in this kind of 2-client,
high-heterogeneity setting among the FedOpt family — that's the reasoning, not a guarantee; the actual
ranking above is what you should report.

**How to describe each algorithm in plain words:**
- **FedAvg** — server just averages the two clients' weights, weighted by how much data each has.
- **FedProx** — same as FedAvg, but during local training each client is penalized for drifting too far from the global model — directly designed for exactly this "very different clients" situation.
- **FedAdam** — the server applies the Adam optimizer to the aggregated update instead of just adding it directly, smoothing out noisy updates caused by the two clients pulling in different directions.
- **FedDistillation (FedDF)** — clients never send weights at all, only their *predictions* on a small dataset both clients share; the server trains the global model to imitate the combined predictions. The only method here that would still work even if the two clients used different model architectures.

The bar chart and per-dataset breakdown above show which algorithm won overall, and whether it did so by
being strong on both datasets or by favoring one over the other — both numbers are worth reporting.